# Prodigal Gene Prediction

![Prodigal Gene Prediction](https://proto-bio.github.io/proto-assets/images/tool/prodigal/hero.png)

This notebook demonstrates `run_prodigal_prediction`, which identifies protein-coding genes in prokaryotic DNA sequences using Prodigal. It covers single-sequence prediction in meta mode, detailed ORF annotation inspection, batch processing, and result export in standard bioinformatics formats.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("prodigal")
display_overview("prodigal")
display_docs_section("prodigal", "Background")

# Prodigal

Prodigal (Prokaryotic Dynamic Programming Genefinding Algorithm) is a fast, reliable gene prediction tool specifically designed for [prokaryotic](https://en.wikipedia.org/wiki/Prokaryote) genomes (bacteria and archaea). It identifies protein-coding genes using dynamic programming, including partial genes at sequence ends, and provides detailed annotations including ribosome binding sites and start codon types.

## Background

**Prokaryotic gene structure:**
Prokaryotic genes are simpler than eukaryotic genes:
- No introns (continuous coding sequence)
- [Ribosome binding site](https://en.wikipedia.org/wiki/Ribosome-binding_site) (RBS) upstream of start codon
- Start codons: ATG (most common), GTG, TTG
- Stop codons: TAA, TAG, TGA

**What Prodigal predicts:**
- Gene boundaries (start and end positions)
- Reading frame and strand
- Start codon type (ATG, GTG, TTG)
- Ribosome binding site motif and spacing
- Partial gene status (truncated at sequence edges)

**Meta vs single-genome mode:**
- **Meta mode**: Uses pre-trained parameters, works on short contigs and mixed samples
- **Single-genome mode**: Trains on input sequence, requires >100kb for reliable training

## Available tools

In [2]:
display_available_tools("prodigal")

- **`run_prodigal_prediction()`** — Prokaryotic ORF and gene prediction using Prodigal

### `run_prodigal_prediction`

`run_prodigal_prediction` predicts protein-coding genes in prokaryotic DNA sequences using Prodigal via pyrodigal Python bindings. It runs in two modes: meta mode uses pre-trained metagenomic parameters and works on short contigs, incomplete assemblies, or mixed microbial samples; single-genome mode trains on the input sequence and requires at least 100 kb. Each predicted gene is returned as an ORF object with strand, reading frame, amino acid translation, nucleotide sequence, 1-indexed coordinates, GC content, start codon type, ribosome binding site motif and spacer, and partial-gene flags. Batch inputs are processed together and results are indexed by input sequence.

In [3]:
from proto_tools import ProdigalInput, ProdigalConfig, ProdigalOutput, run_prodigal_prediction

In [4]:
# Display input docs
display_api_reference("prodigal", "input", "run_prodigal_prediction")

# An E. coli lacZ gene fragment encoding beta-galactosidase
sequence = (
    "ATGACCATGATTACGGATTCACTGGCCGTCGTTTTACAACGTCGTGACTGG"
    "GAAAACCCTGGCGTTACCCAACTTAATCGCCTTGCAGCACATCCCCCTTTC"
    "GCCAGCTGGCGTAATAGCGAAGAGGCCCGCACCGATCGCCCTTCCCAACAG"
    "TTGCGCAGCCTGAATGGCGAATGGCGCTTTGCCTGGTTTCCGGCACCAGAA"
    "GCGGTGCCGGAAAGCTGGCTGGAGTGCGATCTTCCTGAGGCCGATACTGTC"
    "GTCGTCCCCTCAAACTGGCAGATGCACGGTTACGATGCGCCCATCTACACC"
    "AACGTGACCTATCCCATTACGGTCAATCCGCCGTTTGTTCCCACGGAGAAT"
    "CCGACGGGTTGTTACTCGCTCACATTTAATGTTGATGAAAGCTGGCTACAG"
    "GAAGGCCAGACGCGAATTATTTTTGATGGCGTTAACTCGGCGTTTCATCTG"
    "TGGTGCAACGGGCGCTGGGTCGGTTACGGCCAGGACAGTCGTTTGCCGTCT"
    "TAA"
)
inputs = ProdigalInput(input_sequences=sequence)

**Input** — `ProdigalInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `input_sequences` | `list[str]` | required | DNA sequence(s) to analyze for open reading frames |

In [5]:
# Display config docs
display_api_reference("prodigal", "config", "run_prodigal_prediction")

# Meta mode with bacterial translation table | see docs above for all fields
config = ProdigalConfig(
    meta_mode=True,
    translation_table="bacterial",
    closed_ends=False,
)

**Config** — `ProdigalConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cpu'` | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `meta_mode` | `bool` | `True` | Use meta mode for short sequences/fragments (True) or single-genome mode (False) |
| `translation_table` | `Literal['standard', 'vertebrate_mitochondrial', 'yeast_mitochondrial', 'mycoplasma', 'invertebrate_mitochondrial', 'ciliate_nuclear', 'echinoderm_mitochondrial', 'euplotid_nuclear', 'bacterial', 'alternative_yeast_nuclear', 'ascidian_mitochondrial', 'alternative_flatworm_mitochondrial', 'blepharisma_nuclear', 'chlorophycean_mitochondrial', 'trematode_mitochondrial', 'scenedesmus_mitochondrial', 'thraustochytrium_mitochondrial', 'rhabdopleuridae_mitochondrial', 'candidate_division_sr1']` | `'bacterial'` | NCBI genetic code for translation (bacterial = table 11, standard = table 1) |
| `closed_ends` | `bool` | `False` | Prevent genes from running off sequence edges (use True for complete circular genomes) |
| `mask` | `bool` | `False` | Treat runs of N as masked; do not call genes across them |
| `min_gene` | `int` | `90` | Minimum gene length in nt; lower for draft assemblies |
| `num_threads` | `int` | `192` | Number of threads for parallel processing (default: auto-detect all available cores) |

In [6]:
# Run the prediction
result = run_prodigal_prediction(inputs, config)

Running run_prodigal_prediction [00:00]

In [7]:
# Display output docs
display_api_reference("prodigal", "output", "run_prodigal_prediction")

# Inspect predicted genes
print(f"Found {result.num_orfs} predicted gene(s)")
for orf in result.predicted_orfs[0]:
    print(f"  {orf.id}: {orf.nucleotide_start}-{orf.nucleotide_end} ({orf.strand} strand, frame {orf.frame})")

# Inspect rich annotations on the first predicted ORF
orf = result.predicted_orfs[0][0]
print(f"\nGene annotations:")
print(f"  Start type:     {orf.start_type}")
print(f"  RBS motif:      {orf.rbs_motif}")
print(f"  RBS spacer:     {orf.rbs_spacer}")
print(f"  GC content:     {orf.gc_content:.3f}")
print(f"  Partial (5'/3'): {orf.partial_begin}/{orf.partial_end}")
print(f"  Protein length: {orf.amino_acid_length} aa")
print(f"  Protein:        {orf.amino_acid_sequence[:50]}...")

# Demonstrate batch processing
batch_sequences = [
    "ATGCGTAAATAA" * 50,
    "ATGGCATAA" * 50,
    "ATGAAACGT" * 50,
]
batch_result = run_prodigal_prediction(ProdigalInput(input_sequences=batch_sequences))
print(f"\nBatch: {batch_result.num_orfs} total genes across {len(batch_sequences)} sequences")
print(f"Genes per sequence: {batch_result.num_orfs_per_sequence}")

**Output** — `ProdigalOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `predicted_orfs` | `list[list[proto_tools.tools.orf_prediction.orf.ORF]]` | `[]` | List of ORF results per input sequence |

Found 1 predicted gene(s)
  seq_0_gene_1: 1-513 (+ strand, frame 1)

Gene annotations:
  Start type:     Edge
  RBS motif:      None
  RBS spacer:     None
  GC content:     55.361
  Partial (5'/3'): 1/0
  Protein length: 170 aa
  Protein:        MTMITDSLAVVLQRRDWENPGVTQLNRLAAHPPFASWRNSEEARTDRPSQ...


Running run_prodigal_prediction [00:00]


Batch: 3 total genes across 3 sequences
Genes per sequence: [1, 1, 1]


## Export Results

Predicted genes can be exported to GFF (standard gene annotation format), CSV or JSON for tabular analysis, or FASTA format (FAA for protein sequences, FNA for nucleotide sequences) for downstream tools such as BLAST or multiple sequence alignment.

In [8]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export("prodigal_results", export_path=output_dir, file_format="gff")
print(f"Exported GFF to {output_dir / 'prodigal_results.gff'}")

result.export("prodigal_results", export_path=output_dir, file_format="csv")
print(f"Exported CSV to {output_dir / 'prodigal_results.csv'}")

result.export("prodigal_results", export_path=output_dir, file_format="faa")
print(f"Exported protein FASTA to {output_dir / 'prodigal_results.faa'}")

Exported GFF to example_output/prodigal_results.gff


Exported CSV to example_output/prodigal_results.csv
Exported protein FASTA to example_output/prodigal_results.faa
